# Multimodal agent handoff verification

Recipe 100 · Level 4 (Advanced) · `Choice` + `Noul` + images

Sources:

- S02: [Primitives](https://docs.typesafe.ai/primitives)
- S03: [Confidence](https://docs.typesafe.ai/confidence)
- S09: [Images](https://docs.typesafe.ai/images)
- S11: [Models](https://docs.typesafe.ai/models)
- S12: [Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Build an evidence gate between an export-producing agent and an export-analyzing agent. A report saying “done” is only a claim. A synthetic screenshot shows the export panel, a tool record supplies a captured output excerpt, and Python checks whether those observations belong to the current task and attempt. The only permitted downstream operation is a simulated `inspect_export` handoff to `export-analyzer`.

One `Choice` classifies what the report claims. Three independent `Noul` statements ask whether the screenshot shows completion, the tool excerpt supports completion, and the visible artifact identity matches the request. Python combines these judgments with exact freshness, provenance and current-revision checks. All four questions fit in one request because none depends on another answer. An optional later agent would run only after this gate; this notebook never runs that agent.

A [review](../../docs/glossary.md#review) is a deferral for someone to inspect the same decision. Every rejected handoff enters `ReviewQueue`; no retry or repair is attempted. [Provenance](../../docs/glossary.md#provenance) records where evidence or an answer came from. Here collector names are simulated trusted-harness metadata, not authenticated signatures. The screenshots are drawings, not photographs or real browser captures. The committed response fixtures are authored synthetic examples; **not measured live**. After deliberately replacing them with recordings, the runtime header and evaluation identify that recorded sample.


## Setup and run mode

A [replay key](../../docs/glossary.md#replay-key) binds the questions, legitimate context and exact image bytes to a stored answer. A fixture miss stops the run. Set the backend to pinned `jev-1.14.0` with experimental images enabled for the opt-in live path; no long-context flag is needed for these small packets. The environment and recording instructions are in the recipe README.

There are 12 validation examples, 12 held-out test examples and two unscored demonstrations. Each example spends one request containing four questions; the local budget allows 26 requests and does not retry. The live backend also counts every network attempt, including its retries.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import evaluate_outcomes
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, plot_confusion_matrix, run_header, show_answer
from jev_cookbook.vision import run_record

helpers = load_helpers()
examples = load_inputs()
labels = load_labels()
validation = select_split(examples, "validation")
test = select_split(examples, "test")
demos = select_split(examples, "demo")
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
budget = helpers.RequestBudget(26)
results = {}
apply_style()
run_header(100, "Multimodal agent handoff verification", backend=backend, n_examples=24)
check = " [synthetic pipeline check, not a Jev result]"
if backend.mode == "recorded":
    check = f" [recorded model={backend.model}, dates={backend.recorded_dates}]"
elif backend.mode == "live":
    check = " [unrecorded exploration, not a reportable held-out result]"
selection = " [validation selection step]"

## The state

The export requirement names one artifact and revision, with three fixed steps: build, write and validate. `agent_report` and `output_excerpt` are untrusted text. The image can contain hostile text too. These are input evidence, never instructions to the verifier or permission to choose a recipient.

The harness supplies task and attempt identifiers, capture times and collector names. Python keeps the destination, enabled flag, current revision and evaluation clock in `control`, outside model state. A production collector would need to authenticate that envelope and atomically enforce current preconditions at action time; string equality in this simulation supplies neither guarantee. Drawing specifications, split membership and labels stay outside inference.


In [ ]:
demo = demos[0]
state, questions, assets = helpers.build_request(demo.fields)
display(Image(data=Path(demo.fields["screenshot"]).read_bytes()))
print("Request:", state["request"])
print("Agent report:", state["agent_report"])
print("Tool record:", state["tool_record"])
print("Capture envelope:", state["capture"])
print("Model state fields:", sorted(state))

## The questions

Each question concerns one independent claim. A completed report can coexist with unfinished pixels, so the report's `Choice` is never treated as proof. The `uncertain` option explicitly permits an indeterminate report. `Noul` supplies a probability, and its derived certainty is `abs(2*p - 1)`; it has no API confidence field. A high-confidence “false” is strong contradictory evidence, not permission to proceed.

The gate requires the complete report and all three positive claims. No model output selects an action, computes an age, changes a budget or interprets a tool exit code as authority. The candidate confidence thresholds and all action names are fixed in Python.


In [ ]:
for name, question in questions.items():
    print(name, question)
print("Fixed recipients:", helpers.RECIPIENTS)
print("Fixed policy candidates:", helpers.CANDIDATES)

## One answer up close

First inspect the typed answers for the ordinary example. Then compare the second screenshot, where a visible instruction asks the verifier to ignore its checks while validation remains pending. The fixture response rejects the screen-completion claim. This demonstrates how the rule consumes a contradiction; it is not a measurement of resistance to prompt injection.


In [ ]:
for example in demos:
    results[example.id] = budget.decide(backend, example.fields)
    print("Demo:", example.id)
    display(Image(data=Path(example.fields["screenshot"]).read_bytes()))
    for name, answer in results[example.id].answers.items():
        print(name)
        show_answer(answer)

## Python's part

A [confidence gate](../../docs/glossary.md#confidence-gate) accepts a judgment only when its certainty reaches a chosen threshold. Here the minimum certainty across the four judgments must clear the threshold, every claim must be positive, and the exact evidence checks must also pass. The 120-second freshness window is an application precondition, not a model-selected parameter.

Python checks the recipient and operation allowlists, enabled flag, current revision, collector names, task/attempt alignment, artifact identity in the tool record, chronological ordering, freshness, exit code and completed step count. Duplicate attempts and exhausted handoff budgets stop further actions. Nothing can execute text from a screenshot, a report or tool output. The action log holds only the allowlisted artifact reference, revision, task and recipient. A confident wrong visual judgment can still pass these checks; inspect that residual error below.

The following demonstrations use the middle candidate, 0.65, only to illustrate the rule. The policy for scored examples is selected on validation in the next section.


In [ ]:
demo_actions, demo_queue = ActionLog(), ReviewQueue()
for example in demos:
    outcome = helpers.route(
        example.fields,
        results[example.id].answers,
        0.65,
        actions=demo_actions,
        queue=demo_queue,
    )
    print(example.id, outcome)
print("Simulated actions:", demo_actions.to_dicts())
print("Review queue:", demo_queue.to_dicts())

## Evaluation

A [gold label](../../docs/glossary.md#gold-label) is the authored expected decision, kept separate from the model request. Each label states whether the entire evidence packet supports a handoff and explains why. [Coverage](../../docs/glossary.md#coverage) is the fraction actually handed off; [accuracy](../../docs/glossary.md#accuracy) is the fraction of those accepted handoffs that are correct; [risk](../../docs/glossary.md#risk) is their error fraction. Review is abstention and does not count as coverage. We also print agreement of the final handoff/review decision with the expected next step over all examples, so a useful abstention is visible without relabeling it an accepted result.

The finite candidate family is `{0.40, 0.65, 0.85}`. Validation selects the threshold with greatest actual coverage among candidates with zero false handoffs on that validation sample; ties prefer the lower threshold. This sample constraint is not a safety guarantee. Selection uses `evaluate_outcomes` with the rule's actual accepted flags, including every unconditional interlock. No confidence sentinel stands in for a rejected example.

The deterministic baseline uses the same packet: it requires the same exact evidence checks and a fixed green-pixel share of at least 0.60 in the status-banner region. It never reads a label or an authored scene name. This represents a UI color monitor coupled to a structured tool success record. It can miss a valid muted theme and accept a green banner above unfinished steps or a different artifact. Its region and thresholds are fixed before validation.


In [ ]:
for example in validation:
    results[example.id] = budget.decide(backend, example.fields)
threshold, sweep = helpers.select_policy(
    validation, results, labels, objective="max_coverage_without_false_handoff"
)
for row in sweep:
    print(
        f"Validation N={len(validation)}, threshold={row['threshold']:.2f}, "
        f"coverage={row['coverage']:.4f}, accuracy={row['accuracy']:.4f}, "
        f"risk={row['risk']:.4f}, false_handoffs={row['false_handoffs']}"
        f"{selection}{check}"
    )
print(f"Frozen threshold={threshold:.2f}, validation N={len(validation)}{selection}{check}")

### Apply the frozen policy once to test

After selection, the test inputs are decided once and cached. The same cached results support the decision table, aggregate measures and error inspection; there are no additional inference calls for analysis. Model and baseline are compared on the same 12 test packets. No test label changes the selected threshold. An unrecorded live run suppresses held-out metric reporting.


In [ ]:
for example in test:
    results[example.id] = budget.decide(backend, example.fields)
outcomes, actions, queue = helpers.decide_split(test, results, threshold)
baseline = [helpers.pixel_record_baseline(example.fields) for example in test]
gold = [labels[example.id]["handoff_supported"] for example in test]
accepted = [outcome.accepted for outcome in outcomes]
baseline_accepted = [value == helpers.HANDOFF for value in baseline]
if backend.mode == "live":
    print("Unrecorded exploration: held-out reporting is disabled.")
else:
    for name, flags in (
        ("Evidence gate", accepted),
        ("Pixel + record baseline", baseline_accepted),
    ):
        metric = evaluate_outcomes(flags, gold)
        agreement = sum(a == g for a, g in zip(flags, gold, strict=True)) / len(gold)
        false_handoffs = sum(a and not g for a, g in zip(flags, gold, strict=True))
        print(
            f"{name}: test N={len(test)}, accepted={metric.n_answered}, "
            f"review={len(test) - metric.n_answered}, coverage={metric.coverage:.4f}, "
            f"accuracy={metric.accuracy:.4f}, risk={metric.risk:.4f}, "
            f"next_step_agreement={agreement:.4f}, false_handoffs={false_handoffs}{check}"
        )
    for example, outcome, baseline_outcome in zip(test, outcomes, baseline, strict=True):
        print(
            example.id,
            "gold_handoff=",
            labels[example.id]["handoff_supported"],
            "gate=",
            outcome.outcome,
            "baseline=",
            baseline_outcome,
            "reason=",
            outcome.reason,
        )
print("Requests spent:", budget.spent, "; four questions in each request")

In [ ]:
if backend.mode != "live":
    matrix = [[0, 0], [0, 0]]
    for expected, actual in zip(gold, accepted, strict=True):
        matrix[int(expected)][int(actual)] += 1
    print(f"Test N={len(test)}; rows=expected review/handoff, columns=actual review/handoff{check}")
    print(matrix)
    figure = plot_confusion_matrix(
        matrix, ["review", "handoff"], title=f"Handoff decisions, N={len(test)}{check}"
    )
    display(figure)
    import matplotlib.pyplot as plt

    plt.close(figure)

### Inspect a confidently wrong acceptance

In the committed synthetic fixtures, a deliberately wrong held-out response claims the displayed artifact matches even though the image names a different file. The tool record still names the requested artifact. Its certainty is above the frozen threshold, and all exact interlocks pass. This is a false handoff. The cell inspects actual accepted errors from this run; recorded answers may produce a different set, including none. The simulation runs no second agent.


In [ ]:
if backend.mode != "live":
    for example, outcome in zip(test, outcomes, strict=True):
        if outcome.accepted and not labels[example.id]["handoff_supported"]:
            print("Accepted error:", example.id, "certainty=", round(outcome.confidence, 4))
            display(Image(data=Path(example.fields["screenshot"]).read_bytes()))
            print("Requested artifact:", example.fields["request"]["artifact"])
            print("Gold explanation:", labels[example.id]["reason"])
            show_answer(results[example.id].answers["same_artifact"])
    print("Simulated handoffs:", actions.to_dicts())
    print("Review queue:", queue.to_dicts())

## What was and was not measured

The cell below reports the mode, N, and for recorded results the returned model and capture dates. The checked sample contains 24 scored packets and two unscored demonstrations, with 26 screenshots and 26 requests. Every numeric quality result in this committed notebook is a synthetic pipeline check. The committed synthetic run measured no live model quality, inference latency, token usage or billing. With recorded fixtures, the cell preserves any API-reported token usage; unknown quantities remain `None`. Replay duration is never inference latency.

The fixed UI drawings are narrow: no OCR noise, browser variation or real tool authentication is tested. The committed synthetic run deliberately still accepts a wrong artifact identity. The test set is small and hand-authored, and validation's zero-error candidate constraint does not guarantee future handoffs. A real deployment needs authenticated evidence collection, explicit authorization and an atomic revision check at the downstream action boundary. All action entries here have `executed=False`.


In [ ]:
print(
    f"Mode={backend.mode}; N={len(test)} held-out, N={len(validation)} validation; demos={len(demos)}"
)
if backend.mode == "recorded":
    print("Returned model:", backend.model, "; capture dates:", backend.recorded_dates)
elif backend.mode == "live":
    print(
        "Returned model:",
        backend.model,
        "; unrecorded exploration only; no reportable held-out result",
    )
else:
    print("Not measured live: a pipeline check, not a Jev result.")
    print(f"N: {len(test)} held-out; {len(validation)} validation; {len(demos)} demos")
print("Example accounting:", run_record(results[demos[0].id], mode=backend.mode))
print("No inference latency or measured cost was captured.")

## Next steps

Compare the earlier text-only [tool outcome verification](../29-tool-outcome-verification/) recipe to see why adding screenshots creates cross-evidence conflicts rather than simply another success field. Practice uncertainty-aware evaluation with [sentiment classification](../01-sentiment-classification/).

To extend the experiment, author fresh screenshots with shifted layouts, wrong revisions and benign quoted instructions before collecting new recorded answers. Freeze a new policy on validation and use a new held-out set. If an additional judgment depends on a localized region or an earlier answer, make it a separate request with its own budget and recorded replay key; independent questions can stay batched. Use the opt-in recorder in the README to capture the same request triples with returned model, date and usage. Read [Images](https://docs.typesafe.ai/images) and [Models](https://docs.typesafe.ai/models) for the experimental API contracts.
